# SplitFed Demonstration Notebook

This notebook provides a simple interactive setup of the Split Federated Learning (SplitFed) simulator.
We will load the MNIST dataset, partition it among 3 simulated clients, and run a short training loop.

### Step 0: Remote Server Setup

If you are running this notebook on a remote Jupyter server (like Google Colab), run the following cell to clone the repository, install dependencies, and set the correct working directory.

In [2]:
import os, subprocess, sys

repo_dir = "ad-sfl"
repo_url = "https://github.com/tomal66/ad-sfl.git"

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", repo_url], check=True)

os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("Current working directory:", os.getcwd())

Current working directory: C:\Users\BIM\ad-sfl\ad-sfl


In [3]:
import torch
from src.data.datasets import get_datasets
from src.data.partition import partition_data_iid
from src.models.split import ClientModel, ServerModel
from src.core.client import SplitFedClient
from src.core.server import SplitFedServer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Step 1: Load and Partition Data

In [4]:
num_clients = 3
batch_size = 64

print("Loading MNIST dataset...")
train_data, test_data = get_datasets("MNIST")

print("Partitioning data info IID subsets...")
client_datasets = partition_data_iid(train_data, num_clients)
print(f"Data partitioned to {len(client_datasets)} clients.")

Loading MNIST dataset...


100%|█████████████████████████████████████| 9.91M/9.91M [05:20<00:00, 30.9kB/s]
100%|██████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 112kB/s]
100%|█████████████████████████████████████| 1.65M/1.65M [00:34<00:00, 47.4kB/s]
100%|█████████████████████████████████████| 4.54k/4.54k [00:01<00:00, 3.48kB/s]

Partitioning data info IID subsets...
Data partitioned to 3 clients.


### Step 2: Initialize Server and Client Models

In SplitFed, a portion of the network is on the client, and the rest is on the server.

In [5]:
import copy

# Server Model initialization
server_model = ServerModel(in_channels=32, hidden_channels=64, num_classes=10, input_size=(14, 14))
server = SplitFedServer(model=server_model, num_clients=num_clients, device=device)

# Client Models initialization (each client starts with same weights)
base_client_model = ClientModel(in_channels=1, hidden_channels=32)
clients = []
for i in range(num_clients):
    client_model = copy.deepcopy(base_client_model)
    client = SplitFedClient(client_id=i, model=client_model, dataset=client_datasets[i], batch_size=batch_size, device=device)
    clients.append(client)

print("Initialized 1 Server and 3 Clients.")

Initialized 1 Server and 3 Clients.


### Step 3: Simulation Loop

In [ ]:
from src.algorithms import run_sfl_round

epochs = 100

for epoch in range(epochs):
    avg_loss = run_sfl_round(clients, server)
        
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")
        
print("Training simulation complete.")
